# Selecta Quarterly Valuation

This notebook performs the quarterly actuarial valuation for Selecta.
It combines claims and premiums data to produce the key valuation metrics.

**Steps:**
1. Load claims and premiums summaries
2. Calculate loss ratio
3. Calculate technical provisions
4. Produce valuation results summary
5. Export results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import date

## Parameters

In [ ]:
VALUATION_DATE = date(2024, 9, 30)  # Quarter end date
DISCOUNT_RATE = 0.035               # Annual risk-free discount rate
RISK_MARGIN_RATE = 0.06             # Cost of capital rate for risk margin

print(f'Valuation Date:      {VALUATION_DATE}')
print(f'Discount Rate:       {DISCOUNT_RATE:.1%}')
print(f'Risk Margin Rate:    {RISK_MARGIN_RATE:.1%}')

## 1. Load Claims and Premiums Summaries

These are produced by `claims.ipynb` and `premiums.ipynb` respectively.

In [ ]:
# In production these would be loaded from output files:
# claims_summary = pd.read_csv('output/claims_summary.csv')
# premiums_summary = pd.read_csv('output/premiums_summary.csv')

# Placeholder data consistent with claims.ipynb and premiums.ipynb outputs
np.random.seed(42)

claims_summary = pd.DataFrame({
    'claim_type': ['Critical Illness', 'Death', 'Disability'],
    'count': [35, 30, 35],
    'total_paid': [1_250_000.0, 2_100_000.0, 980_000.0],
    'total_ibnr': [125_000.0, 95_000.0, 110_000.0],
    'total_incurred': [1_375_000.0, 2_195_000.0, 1_090_000.0],
})

premiums_summary = pd.DataFrame({
    'product': ['Critical Illness', 'Income Protection', 'Term Life', 'Whole Life'],
    'policy_count': [45, 38, 72, 25],
    'total_annual_premium': [540_000.0, 418_000.0, 864_000.0, 375_000.0],
})
premiums_summary['quarterly_earned'] = premiums_summary['total_annual_premium'] / 4

print('Claims summary:')
print(claims_summary.to_string(index=False))
print('\nPremiums summary:')
print(premiums_summary.to_string(index=False))

## 2. Calculate Loss Ratio

In [ ]:
total_incurred = claims_summary['total_incurred'].sum()
total_earned_premium = premiums_summary['quarterly_earned'].sum()

loss_ratio = total_incurred / total_earned_premium

print(f'Total Incurred Claims:    £{total_incurred:,.2f}')
print(f'Total Earned Premium:     £{total_earned_premium:,.2f}')
print(f'Loss Ratio:               {loss_ratio:.1%}')

## 3. Calculate Technical Provisions

In [ ]:
# Best estimate liability (BEL) — sum of IBNR reserves
bel = claims_summary['total_ibnr'].sum()

# Simplified risk margin: cost-of-capital approach
# Risk margin = risk_margin_rate * SCR, where SCR ≈ 25% of BEL (simplified)
scr = 0.25 * bel
risk_margin = RISK_MARGIN_RATE * scr / (1 + DISCOUNT_RATE)

# Technical provisions = BEL + risk margin
technical_provisions = bel + risk_margin

print(f'Best Estimate Liability (BEL):  £{bel:,.2f}')
print(f'Solvency Capital Requirement:   £{scr:,.2f}')
print(f'Risk Margin:                    £{risk_margin:,.2f}')
print(f'Technical Provisions:           £{technical_provisions:,.2f}')

## 4. Valuation Results Summary

In [ ]:
results = pd.DataFrame({
    'Metric': [
        'Valuation Date',
        'Active Policies',
        'Total Claims (Quarter)',
        'Total Earned Premium (Quarter)',
        'Loss Ratio',
        'Best Estimate Liability',
        'Risk Margin',
        'Technical Provisions',
    ],
    'Value': [
        str(VALUATION_DATE),
        f"{premiums_summary['policy_count'].sum()}",
        f"£{total_incurred:,.2f}",
        f"£{total_earned_premium:,.2f}",
        f"{loss_ratio:.1%}",
        f"£{bel:,.2f}",
        f"£{risk_margin:,.2f}",
        f"£{technical_provisions:,.2f}",
    ],
})

print('\n=== QUARTERLY VALUATION RESULTS ===')
print(results.to_string(index=False))

## 5. Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Incurred claims by type
claims_summary.plot.bar(
    x='claim_type', y='total_incurred', ax=axes[0],
    legend=False, color='coral'
)
axes[0].set_title('Incurred Claims by Type')
axes[0].set_xlabel('Claim Type')
axes[0].set_ylabel('Amount (£)')
axes[0].tick_params(axis='x', rotation=0)

# Technical provisions breakdown (stacked bar)
tp_data = pd.DataFrame({
    'Component': ['BEL', 'Risk Margin'],
    'Amount': [bel, risk_margin],
})
tp_data.plot.bar(x='Component', y='Amount', ax=axes[1], legend=False, color=['steelblue', 'orange'])
axes[1].set_title('Technical Provisions Breakdown')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Amount (£)')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle(f'Selecta Quarterly Valuation — {VALUATION_DATE}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Export Results

In [ ]:
# os.makedirs('output', exist_ok=True)
# results.to_csv(f'output/quarterly_valuation_{VALUATION_DATE}.csv', index=False)
print('Quarterly valuation complete.')